In [ ]:
from pathlib import Path
import os
import zipfile
import urllib.request
import shutil

GITHUB_USER = "Facuriy"
GITHUB_REPO = "agro-ai-kurs"
BRANCH = "main"
BLOCK_DIR = Path("02_pflanzenzaehlung_computer_vision_DE")
ZIP_NAME = "02_pflanzenzaehlung_computer_vision_DE.zip"
ZIP_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}/raw/{BRANCH}/{ZIP_NAME}"
RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{BRANCH}/02_pflanzenzaehlung_computer_vision_DE"
DATA_FILE = Path("class_data_plants/examples.csv")
DATA_FILES = ['class_data_plants/beispiel_1_rgb.png',
    'class_data_plants/beispiel_1_rgb_ground_truth.csv',
    'class_data_plants/beispiel_2_altum_multispektral.png',
    'class_data_plants/beispiel_2_altum_multispektral_ground_truth.csv',
    'class_data_plants/beispiel_3_rededge_multispektral.png',
    'class_data_plants/beispiel_3_rededge_multispektral_ground_truth.csv',
    'class_data_plants/beispiel_4_m3multi_multispektral.png',
    'class_data_plants/beispiel_4_m3multi_multispektral_ground_truth.csv',
    'class_data_plants/examples.csv']

def extract_zip(zip_path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        for member in names:
            normalized = Path(member.replace("\\", "/"))
            target = Path(".") / normalized
            if member.endswith("/"):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as source, open(target, "wb") as dest:
                    shutil.copyfileobj(source, dest)
        print("ZIP entpackt:", len(names), "Dateien")

def in_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False

def find_course_folder():
    candidates = [Path("."), BLOCK_DIR]
    candidates += [p.parent.parent for p in Path(".").rglob("examples.csv")]
    for candidate in candidates:
        if (candidate / DATA_FILE).exists():
            return candidate
    return None

def download_direct_files():
    for rel in DATA_FILES:
        target = Path(rel)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            url = f"{RAW_BASE}/{rel}"
            print("Lade Datei:", url)
            urllib.request.urlretrieve(url, target)

course_folder = find_course_folder()

if course_folder is None and in_colab():
    print("Lade Unterrichtsdaten:", ZIP_URL)
    urllib.request.urlretrieve(ZIP_URL, ZIP_NAME)
    try:
        extract_zip(ZIP_NAME)
    except zipfile.BadZipFile:
        print("ZIP konnte nicht gelesen werden. Direkter Download wird benutzt.")
    course_folder = find_course_folder()

if course_folder is None and in_colab():
    download_direct_files()
    course_folder = find_course_folder()

if course_folder is None:
    found = [str(p) for p in Path(".").rglob("*")][:30]
    raise FileNotFoundError("Die Unterrichtsdaten wurden nicht gefunden. Gefundene Dateien: " + str(found))

os.chdir(course_folder)

print("Arbeitsordner:", Path.cwd())
print("Datensatz:", DATA_FILE)


# Pflanzenzählung mit Computer Vision

Dieses Notebook ist für den Kurs vorbereitet. Es nutzt kleine, schnelle Unterrichtsdaten.

## Hinweise zur Übung

**Ziel der Einheit:** Die Studierenden sollen eine klassische Computer-Vision-Pipeline verstehen: Bild laden, Pflanzen hervorheben, Maske bilden, Objekte zählen, Ergebnis bewerten.

**Empfohlener Ablauf (60-90 min):**

1. Erst nur RGB und Excess Green erklären.
2. Dann Schwellenwert, Mindestfläche und Glättung live verändern.
3. Ein gutes und ein schlechtes Beispiel vergleichen.
4. Precision/Recall/F1 anhand von TP, FP und FN interpretieren.

**Didaktischer Punkt:** Ein einfacher Algorithmus ist transparent. Gerade wenn er scheitert, lernen die Studierenden viel über Annahmen, Bildqualität und biologische Variabilität.

## Erwartete Beobachtungen und Diskussion

**Typische Interpretation der Metriken:**

- Hohe Precision, niedriger Recall: Der Algorithmus ist streng und übersieht Pflanzen.
- Niedrige Precision, hoher Recall: Der Algorithmus erkennt viel, aber auch viele Fehlalarme.
- F1 ist gut für den Vergleich, ersetzt aber nicht den Blick auf Overlay und Fehlerbeispiele.

**Diskussionsfragen:**

- Warum ist das einfache RGB-Beispiel nicht automatisch das beste?
- Welche Fehler entstehen bei überlappenden Pflanzen?
- Warum braucht man Ground Truth für ehrliche Bewertung?

# Pflanzenzählung mit Python

Eine praktische Einführung in Bildanalyse für Studierende der Agronomie.

In diesem Notebook laden wir kleine echte Bildausschnitte aus UAV-Orthomosaiken,
segmentieren Pflanzen mit einfachen Regeln und bewerten die Ergebnisse mit
Precision, Recall und F1.

## 1. Motivation: Warum Pflanzen automatisch zählen?

Pflanzenzählung hilft bei Feldaufgang, Bestandesdichte, Sortenversuchen,
Stressmonitoring und Managemententscheidungen. Programmierung ist hier kein
Selbstzweck: Wir übersetzen ein agronomisches Problem in wiederholbare Schritte.

### Mini-Glossar: Was passiert in dieser Klasse?

| Begriff | Einfache Erkl?rung |
|---|---|
| **Pixel** | Ein einzelner Bildpunkt. Jedes Pixel hat Farbwerte, z. B. Rot, Gr?n und Blau. |
| **RGB-Bild** | Ein normales Farbbild mit drei Kan?len: Red, Green, Blue. |
| **Multispektralbild** | Ein Bild mit zus?tzlichen Kan?len, z. B. RedEdge oder NIR. Diese Kan?le k?nnen Pflanzenzustand sichtbar machen, den das Auge kaum sieht. |
| **Segmentierung** | Wir trennen Bildbereiche: Pflanze oder Nicht-Pflanze. |
| **Maske** | Ein Schwarz-Wei?-Bild: wei?/True = geh?rt zur Pflanze, schwarz/False = Hintergrund. |
| **Objekt / Komponente** | Eine zusammenh?ngende Gruppe von Pflanzen-Pixeln. Daraus sch?tzen wir eine Pflanze. |
| **Zentroid** | Mittelpunkt eines erkannten Objekts. Er wird als Pflanzenposition angezeigt. |

Die Klasse folgt einer einfachen Computer-Vision-Pipeline:

```text
Bild laden -> Pflanzenfarbe hervorheben -> Maske erzeugen -> Objekte finden -> Pflanzen z?hlen -> Ergebnis bewerten
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage import exposure, filters, measure, morphology

DATA_DIR = Path("class_data_plants")
OUT_DIR = Path("class_outputs_plants")
OUT_DIR.mkdir(exist_ok=True)

examples_path = DATA_DIR / "examples.csv"
if not examples_path.exists():
    raise FileNotFoundError(f"Datei fehlt: {examples_path}")

examples = pd.read_csv(examples_path)
examples

## 2. Bilddaten laden und verstehen

Wir arbeiten mit kleinen PNG-Dateien. Die originalen Orthomosaike sind viel
größer; für eine Unterrichtsstunde reicht ein repräsentativer Ausschnitt.

In [ ]:
def load_rgb(example_row):
    path = DATA_DIR / example_row["image_file"]
    if not path.exists():
        raise FileNotFoundError(path)
    return np.array(Image.open(path).convert("RGB"))


fig, axes = plt.subplots(1, len(examples), figsize=(4 * len(examples), 4))
for ax, (_, row) in zip(axes, examples.iterrows()):
    img = load_rgb(row)
    ax.imshow(img)
    ax.set_title(f"{row['example_id']}\n{row['sensor']}\n{row['difficulty']}")
    ax.axis("off")
plt.tight_layout()

## 3. Einfache Vorverarbeitung

Viele Pflanzen sind grüner als der Boden. Eine einfache Kennzahl ist
`Excess Green`: `2*G - R - B`. Das ist kein perfektes Modell, aber sehr gut
geeignet, um Bildanalyse zu verstehen.

### Definition: Excess Green

F?r junge gr?ne Pflanzen reicht oft ein sehr einfacher Farbindex:

$$\text{ExG} = 2 \cdot G - R - B$$

Dabei sind `R`, `G` und `B` die roten, gr?nen und blauen Farbwerte. Wenn ein Pixel deutlich gr?ner ist als rot/blau, wird `ExG` gro?.

**Wichtig:** Das ist keine biologische Wahrheit. Es ist eine einfache Regel, die bei gr?nen Pflanzen gut funktionieren kann, aber bei Schatten, trockenem Boden, gelben Bl?ttern oder multispektralen Kompositbildern scheitern kann.

In [ ]:
def excess_green(rgb):
    arr = rgb.astype("float32") / 255.0
    r, g, b = arr[..., 0], arr[..., 1], arr[..., 2]
    exg = 2 * g - r - b
    return exposure.rescale_intensity(exg, in_range="image", out_range=(0, 1))


row = examples.iloc[0]
img = load_rgb(row)
exg = excess_green(img)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img)
axes[0].set_title("RGB")
axes[1].imshow(exg, cmap="gray")
axes[1].set_title("Excess Green")
axes[2].hist(exg.ravel(), bins=50, color="green")
axes[2].set_title("Histogramm")
for ax in axes[:2]:
    ax.axis("off")
plt.tight_layout()

## 4. Einfache Pflanzendetektion

Die Funktion unten macht vier Dinge:

1. Grünheitsindex berechnen
2. Bild glätten
3. Schwellenwert anwenden
4. Kleine Objekte entfernen und Pflanzenzentren bestimmen

### Was machen Threshold, Morphologie und Connected Components?

1. **Threshold / Schwellenwert:** Wir entscheiden: `ExG > Schwelle` bedeutet Pflanze.
2. **Gl?ttung:** Kleine St?rungen werden reduziert, damit einzelne Pixel nicht zu falschen Pflanzen werden.
3. **Morphologie:** Kleine Objekte werden entfernt und kleine L?cher geschlossen.
4. **Connected Components:** Zusammenh?ngende wei?e Bereiche in der Maske werden als einzelne Objekte gez?hlt.
5. **Fl?chenfilter:** Zu kleine Objekte sind oft Rauschen; zu gro?e Objekte sind oft zusammengewachsene Pflanzen oder Fehler.

Diese Pipeline ist bewusst simpel. Genau deshalb ist sie gut für eine erste Programmierklasse: Man sieht sofort, welche Annahmen der Algorithmus macht.

In [ ]:
def detect_plants(rgb, threshold=0.08, min_area_px=18, max_area_px=900, smoothing=1.0):
    exg = excess_green(rgb)
    smooth = filters.gaussian(exg, sigma=smoothing)
    mask = smooth > threshold
    mask = morphology.remove_small_objects(mask, min_size=max(1, int(min_area_px)))
    mask = morphology.remove_small_holes(mask, area_threshold=31)
    mask = morphology.opening(mask, morphology.disk(1))

    labels = measure.label(mask)
    rows = []
    keep = np.zeros_like(mask, dtype=bool)
    for region in measure.regionprops(labels):
        if min_area_px <= region.area <= max_area_px:
            y, x = region.centroid
            rows.append({"x": x, "y": y, "area_px": region.area})
            keep[labels == region.label] = True
    return keep, pd.DataFrame(rows)

## 5. AUSPROBIEREN: Parameter ausprobieren

Ändern Sie die Werte und führen Sie die nächsten Zellen erneut aus.

In [ ]:
# AUSPROBIEREN
EXAMPLE_ID = examples.iloc[0]["example_id"]
THRESHOLD = float(examples.loc[examples.example_id == EXAMPLE_ID, "suggested_threshold"].iloc[0])
MIN_AREA_PX = int(examples.loc[examples.example_id == EXAMPLE_ID, "min_area_px"].iloc[0])
MAX_AREA_PX = int(examples.loc[examples.example_id == EXAMPLE_ID, "max_area_px"].iloc[0])
SMOOTHING = 1.0

print(EXAMPLE_ID, THRESHOLD, MIN_AREA_PX, MAX_AREA_PX)

In [ ]:
row = examples.loc[examples.example_id == EXAMPLE_ID].iloc[0]
img = load_rgb(row)
mask, pred = detect_plants(img, THRESHOLD, MIN_AREA_PX, MAX_AREA_PX, SMOOTHING)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(img)
axes[0].set_title("Bild")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Pflanzenmaske")
axes[2].imshow(img)
if len(pred):
    axes[2].scatter(pred.x, pred.y, s=22, c="yellow", marker="+")
axes[2].set_title(f"Detektionen: {len(pred)}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## 6. Metriken: Wie gut ist die Zählung?

Wir vergleichen die gefundenen Zentren mit manuell markierten Referenzpunkten.

- **TP**: Pflanze korrekt gefunden
- **FP**: Detektion, aber keine Referenzpflanze in der Nähe
- **FN**: Referenzpflanze wurde übersehen
- **Precision**: Wie sauber sind die Detektionen?
- **Recall**: Wie viele echte Pflanzen wurden gefunden?
- **F1**: Kompromiss aus Precision und Recall

### Definitionen der Metriken

Wir vergleichen erkannte Pflanzenzentren mit Referenzpunkten (*Ground Truth*).

| Metrik | Bedeutung |
|---|---|
| **TP** (*True Positive*) | Eine echte Pflanze wurde richtig erkannt. |
| **FP** (*False Positive*) | Der Algorithmus meldet eine Pflanze, dort ist aber keine Referenzpflanze. |
| **FN** (*False Negative*) | Eine echte Pflanze wurde ?bersehen. |
| **Precision** | Von allen erkannten Pflanzen: Wie viele waren wirklich richtig?  $$Precision = \frac{TP}{TP + FP}$$ |
| **Recall** | Von allen echten Pflanzen: Wie viele wurden gefunden?  $$Recall = \frac{TP}{TP + FN}$$ |
| **F1** | Harmonischer Mittelwert aus Precision und Recall. Gut, wenn beide wichtig sind.  $$F1 = \frac{2 \cdot Precision \cdot Recall}{Precision + Recall}$$ |
| **Count Error** | Differenz zwischen vorhergesagter und echter Pflanzenzahl. |

**Interpretation:**
- Hohe Precision, niedriger Recall: Der Algorithmus ist vorsichtig, ?bersieht aber Pflanzen.
- Niedrige Precision, hoher Recall: Der Algorithmus findet viele Pflanzen, aber auch viele falsche.
- F1 ist praktisch, wenn wir einen einzelnen Vergleichswert brauchen.

In [ ]:
def load_ground_truth(example_row):
    path = DATA_DIR / example_row["ground_truth_file"]
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def detection_metrics(pred, gt, tolerance_px=18):
    pred_xy = pred[["x", "y"]].to_numpy(float) if len(pred) else np.empty((0, 2))
    gt_xy = gt[["x", "y"]].to_numpy(float) if len(gt) else np.empty((0, 2))
    used = set()
    tp = 0
    for gx, gy in gt_xy:
        if len(pred_xy) == 0:
            continue
        d = np.sqrt((pred_xy[:, 0] - gx) ** 2 + (pred_xy[:, 1] - gy) ** 2)
        for idx in np.argsort(d):
            if idx not in used and d[idx] <= tolerance_px:
                used.add(int(idx))
                tp += 1
                break
    fp = len(pred_xy) - tp
    fn = len(gt_xy) - tp
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return pd.Series({
        "Referenz": len(gt_xy), "Detektionen": len(pred_xy),
        "TP": tp, "FP": fp, "FN": fn,
        "Precision": precision, "Recall": recall, "F1": f1,
        "Zählfehler": len(pred_xy) - len(gt_xy),
    })


gt = load_ground_truth(row)
metrics = detection_metrics(pred, gt)
metrics

In [ ]:
plt.figure(figsize=(7, 7))
plt.imshow(img)
plt.scatter(gt.x, gt.y, s=44, facecolors="none", edgecolors="white", label="Referenz")
if len(pred):
    plt.scatter(pred.x, pred.y, s=24, c="yellow", marker="+", label="Detektion")
plt.title(f"{EXAMPLE_ID}: Referenz vs. Detektion")
plt.axis("off")
plt.legend()
plt.tight_layout()

## 7. Alle Bilder vergleichen

In [ ]:
results = []
for _, row in examples.iterrows():
    img = load_rgb(row)
    gt = load_ground_truth(row)
    mask, pred = detect_plants(
        img,
        threshold=float(row.suggested_threshold),
        min_area_px=int(row.min_area_px),
        max_area_px=int(row.max_area_px),
        smoothing=1.0,
    )
    m = detection_metrics(pred, gt)
    m["example_id"] = row.example_id
    m["sensor"] = row.sensor
    m["difficulty"] = row.difficulty
    results.append(m)

results = pd.DataFrame(results)
cols = ["example_id", "sensor", "difficulty", "Referenz", "Detektionen", "Precision", "Recall", "F1", "Zählfehler"]
results[cols]

In [ ]:
ax = results.set_index("example_id")[["Precision", "Recall", "F1"]].plot(kind="bar", figsize=(10, 4))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Wert")
ax.set_title("Vergleich der Metriken")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

## 8. Grenzen der Methode

Diese einfache Methode ist absichtlich transparent, aber nicht perfekt.

- Schatten können wie Pflanzen wirken.
- Überlappende Pflanzen werden oft als eine Pflanze gezählt.
- Bodenfarbe und Pflanzenstadium ändern den besten Schwellenwert.
- Multispektrale Daten enthalten mehr Information, brauchen aber saubere Bandinterpretation.
- Für echte Feldversuche braucht man robuste Validierung mit unabhängigen Bildern.

## 9. Ausblick

Der nächste Schritt wäre ein automatischer Algorithmus, der Parameter pro Sensor
kalibriert, oder ein kleines Machine-Learning-Modell. Wichtig bleibt: Erst das
agronomische Ziel definieren, dann die Metrik wählen.